# Event sourcing: the state at each point in time

The photo service stores each change to a photo as an event in an append-only log, instead of
changing a row in a table. The current state (and any older state) is computed from the
events.

In [1]:
import json
import sqlite3

import pandas as pd

pd.set_option("display.max_colwidth", None)

## 1. An append-only event log

The event store is a table that rejects each `UPDATE` and `DELETE`: events are only added.

In [2]:
db = sqlite3.connect(":memory:")
db.executescript("""
CREATE TABLE events (seq INTEGER PRIMARY KEY, time TEXT, type TEXT, photo_id INTEGER, data TEXT);
CREATE TRIGGER no_update BEFORE UPDATE ON events BEGIN SELECT RAISE(ABORT, 'append only'); END;
CREATE TRIGGER no_delete BEFORE DELETE ON events BEGIN SELECT RAISE(ABORT, 'append only'); END;
""")


def record(time: str, type: str, photo_id: int, **data) -> None:
    db.execute(
        "INSERT INTO events (time, type, photo_id, data) VALUES (?, ?, ?, ?)",
        (time, type, photo_id, json.dumps(data)),
    )


record(
    "2021-12-03T09:18:32.124Z", "addPhoto", 133422131, user=54351, path="/st/u211/1U6uFl47Fy.jpg"
)
record(
    "2021-12-03T09:18:32.129Z", "addPhoto", 133422132, user=13221, path="/st/u11b/MFxlL1FY8V.jpg"
)
record("2021-12-03T09:18:32.131Z", "addPhoto", 133422133, user=54351, path="/st/x81/ITzhcSmv9s.jpg")
record("2021-12-03T10:02:11.508Z", "updatePhotoData", 133422131, user=54351, title="Sunset")
record(
    "2021-12-04T08:31:02.113Z",
    "replacePhoto",
    133422131,
    user=54351,
    path="/st/x594/vipxBMFlLF.jpg",
    operation="/filter/palma",
)
record(
    "2021-12-04T08:31:47.920Z",
    "replacePhoto",
    133422133,
    user=54351,
    path="/st/y20/Qm3kd9LpXa.jpg",
    operation="/filter/mono",
)
record(
    "2021-12-04T08:32:22.045Z",
    "replacePhoto",
    133422133,
    user=54351,
    path="/st/x81/ITzhcSmv9s.jpg",
    operation="/undo",
)
record("2021-12-05T17:45:09.377Z", "updatePhotoData", 133422132, user=13221, title="Winter walk")
record("2021-12-06T21:14:55.602Z", "deletePhoto", 133422131, user=54351)
record("2021-12-07T07:02:30.219Z", "updatePhotoData", 133422133, user=54351, title="Birthday cake")

events = pd.read_sql("SELECT * FROM events", db)


def notation(e) -> str:
    args = ", ".join(f"{k}={json.dumps(v)}" for k, v in json.loads(e.data).items())
    return f"{e.type}(id={e.photo_id}, {args})"


events.assign(event=[notation(e) for e in events.itertuples()])[["seq", "time", "event"]]

,seq,time,event
0,1,2021-12-03T09:18:32.124Z,"addPhoto(id=133422131, user=54351, path=""/st/u211/1U6uFl47Fy.jpg"")"
1,2,2021-12-03T09:18:32.129Z,"addPhoto(id=133422132, user=13221, path=""/st/u11b/MFxlL1FY8V.jpg"")"
2,3,2021-12-03T09:18:32.131Z,"addPhoto(id=133422133, user=54351, path=""/st/x81/ITzhcSmv9s.jpg"")"
3,4,2021-12-03T10:02:11.508Z,"updatePhotoData(id=133422131, user=54351, title=""Sunset"")"
4,5,2021-12-04T08:31:02.113Z,"replacePhoto(id=133422131, user=54351, path=""/st/x594/vipxBMFlLF.jpg"", operation=""/filter/palma"")"
5,6,2021-12-04T08:31:47.920Z,"replacePhoto(id=133422133, user=54351, path=""/st/y20/Qm3kd9LpXa.jpg"", operation=""/filter/mono"")"
6,7,2021-12-04T08:32:22.045Z,"replacePhoto(id=133422133, user=54351, path=""/st/x81/ITzhcSmv9s.jpg"", operation=""/undo"")"
7,8,2021-12-05T17:45:09.377Z,"updatePhotoData(id=133422132, user=13221, title=""Winter walk"")"
8,9,2021-12-06T21:14:55.602Z,"deletePhoto(id=133422131, user=54351)"
9,10,2021-12-07T07:02:30.219Z,"updatePhotoData(id=133422133, user=54351, title=""Birthday cake"")"


In [3]:
try:
    db.execute("UPDATE events SET data = '{}' WHERE seq = 4")
except sqlite3.DatabaseError as e:
    print("UPDATE rejected:", e)

UPDATE rejected: append only


## 2. The current state is computed from all events

`apply` changes the state for one event; `replay` applies all events in order.

In [4]:
def apply(state: dict, e) -> dict:
    data = json.loads(e.data)
    if e.type == "deletePhoto":
        return {p: photo for p, photo in state.items() if p != e.photo_id}
    photo = dict(state.get(e.photo_id, {}))
    if e.type == "addPhoto":
        photo = {"user": data["user"], "path": data["path"], "title": ""}
    elif e.type == "updatePhotoData":
        photo["title"] = data["title"]
    elif e.type == "replacePhoto":
        photo["path"] = data["path"]
    return state | {e.photo_id: photo}


def replay(events: pd.DataFrame, state: dict | None = None) -> dict:
    state = dict(state or {})
    for e in events.itertuples():
        state = apply(state, e)
    return state


def table(state: dict) -> pd.DataFrame:
    return pd.DataFrame.from_dict(state, orient="index").rename_axis("photo_id")


table(replay(events))

,user,path,title
photo_id,,,
133422132,13221,/st/u11b/MFxlL1FY8V.jpg,Winter walk
133422133,54351,/st/x81/ITzhcSmv9s.jpg,Birthday cake


## 3. The state after each event

Each row is the state after one more event (`·`: the photo does not exist at this point).

In [5]:
PHOTOS = [133422131, 133422132, 133422133]


def describe(photo: dict | None) -> str:
    if photo is None:
        return "·"
    return f"{photo['title'] or '(no title)'}, {photo['path'].split('/')[-1]}"


rows, state = [], {}
for e in events.itertuples():
    state = apply(state, e)
    rows.append(
        {"seq": e.seq, "event": f"{e.type}({e.photo_id})"}
        | {p: describe(state.get(p)) for p in PHOTOS}
    )
pd.DataFrame(rows).set_index("seq")

,event,133422131,133422132,133422133
seq,,,,
1,addPhoto(133422131),"(no title), 1U6uFl47Fy.jpg",·,·
2,addPhoto(133422132),"(no title), 1U6uFl47Fy.jpg","(no title), MFxlL1FY8V.jpg",·
3,addPhoto(133422133),"(no title), 1U6uFl47Fy.jpg","(no title), MFxlL1FY8V.jpg","(no title), ITzhcSmv9s.jpg"
4,updatePhotoData(133422131),"Sunset, 1U6uFl47Fy.jpg","(no title), MFxlL1FY8V.jpg","(no title), ITzhcSmv9s.jpg"
5,replacePhoto(133422131),"Sunset, vipxBMFlLF.jpg","(no title), MFxlL1FY8V.jpg","(no title), ITzhcSmv9s.jpg"
6,replacePhoto(133422133),"Sunset, vipxBMFlLF.jpg","(no title), MFxlL1FY8V.jpg","(no title), Qm3kd9LpXa.jpg"
7,replacePhoto(133422133),"Sunset, vipxBMFlLF.jpg","(no title), MFxlL1FY8V.jpg","(no title), ITzhcSmv9s.jpg"
8,updatePhotoData(133422132),"Sunset, vipxBMFlLF.jpg","Winter walk, MFxlL1FY8V.jpg","(no title), ITzhcSmv9s.jpg"
9,deletePhoto(133422131),·,"Winter walk, MFxlL1FY8V.jpg","(no title), ITzhcSmv9s.jpg"


## 4. The state at any time

Replaying only the events up to a given time gives the state at that time, for example the
photo that was deleted later.

In [6]:
def state_at(time: str) -> pd.DataFrame:
    return table(replay(events[events.time <= time]))


state_at("2021-12-05T00:00:00Z")

,user,path,title
photo_id,,,
133422131,54351,/st/x594/vipxBMFlLF.jpg,Sunset
133422132,13221,/st/u11b/MFxlL1FY8V.jpg,
133422133,54351,/st/x81/ITzhcSmv9s.jpg,


## 5. Several views from the same events

Different parts of the system compute different views from the same log.

In [7]:
now = table(replay(events))
now.groupby("user").size().rename("photos now")

user
13221    1
54351    1
Name: photos now, dtype: int64

In [8]:
edits = events[events.type == "replacePhoto"].assign(
    operation=lambda d: d.data.map(lambda x: json.loads(x)["operation"])
)
edits[["time", "photo_id", "operation"]]

,time,photo_id,operation
4,2021-12-04T08:31:02.113Z,133422131,/filter/palma
5,2021-12-04T08:31:47.920Z,133422133,/filter/mono
6,2021-12-04T08:32:22.045Z,133422133,/undo


In [9]:
deleted = events[events.type == "deletePhoto"].photo_id.tolist()
undone = edits[edits.operation == "/undo"].photo_id.tolist()
print(f"deleted photos: {deleted}")
print(f"photos where the user tried a filter and undid it: {undone}")

deleted photos: [133422131]
photos where the user tried a filter and undid it: [133422133]


## 6. The same changes in a mutable table

A normal table has the same current state. But each `UPDATE` and `DELETE` removes the old
value, so the questions of section 5 cannot be answered from it.

In [10]:
mutable = sqlite3.connect(":memory:")
mutable.execute(
    "CREATE TABLE photos (photo_id INTEGER PRIMARY KEY, user INT, path TEXT, title TEXT)"
)
for e in events.itertuples():
    d = json.loads(e.data)
    if e.type == "addPhoto":
        mutable.execute(
            "INSERT INTO photos VALUES (?, ?, ?, '')", (e.photo_id, d["user"], d["path"])
        )
    elif e.type == "updatePhotoData":
        mutable.execute("UPDATE photos SET title = ? WHERE photo_id = ?", (d["title"], e.photo_id))
    elif e.type == "replacePhoto":
        mutable.execute("UPDATE photos SET path = ? WHERE photo_id = ?", (d["path"], e.photo_id))
    elif e.type == "deletePhoto":
        mutable.execute("DELETE FROM photos WHERE photo_id = ?", (e.photo_id,))
pd.read_sql("SELECT * FROM photos", mutable).set_index("photo_id")

,user,path,title
photo_id,,,
133422132,13221,/st/u11b/MFxlL1FY8V.jpg,Winter walk
133422133,54351,/st/x81/ITzhcSmv9s.jpg,Birthday cake


## 7. Snapshots

To make reads faster, store the state after some event (a snapshot) and replay only the
later events. The snapshot is a copy of the full state.

In [11]:
snapshot = {"seq": 6, "state": replay(events[events.seq <= 6])}
later = events[events.seq > snapshot["seq"]]
print(f"full replay: {len(events)} events")
n = len(snapshot["state"])
print(f"snapshot after event {snapshot['seq']} (a copy of {n} photos) + {len(later)} events")
print("same state:", replay(later, snapshot["state"]) == replay(events))

full replay: 10 events
snapshot after event 6 (a copy of 3 photos) + 4 events
same state: True


## 8. Drawbacks: the log grows, and deleted data stays in it

In [12]:
print(f"{len(events)} events for {len(now)} photos that exist now")
events[events.photo_id == 133422131][["seq", "type", "data"]]

10 events for 2 photos that exist now


,seq,type,data
0,1,addPhoto,"{""user"": 54351, ""path"": ""/st/u211/1U6uFl47Fy.jpg""}"
3,4,updatePhotoData,"{""user"": 54351, ""title"": ""Sunset""}"
4,5,replacePhoto,"{""user"": 54351, ""path"": ""/st/x594/vipxBMFlLF.jpg"", ""operation"": ""/filter/palma""}"
8,9,deletePhoto,"{""user"": 54351}"
